In [7]:
import urllib.request
from io import StringIO
from pathlib import Path

import pandas as pd

HELP_URL = "https://api.erg.ic.ac.uk/AirQuality/help"   # The API page

# Creates a path object for the snapshot, which provides methods later for existence, parents, reading
HELP_SNAPSHOT = Path("snapshots/help.html")


# Download once - afterwards always read the saved copy.
# This is "being kind" to the API :)
#
# If the file path does not exist, make a directory named after the parent of the file path (snapshots).
# Then open the URL and write the exact bytes of the response to the file path.

if not HELP_SNAPSHOT.exists():                          # This checks if the html file exists in the snapshot folder
    HELP_SNAPSHOT.parent.mkdir(exist_ok=True)           # This first points at the parent directory - snapshots - and then makes a folder called snapshots
    with urllib.request.urlopen(HELP_URL) as response:
        HELP_SNAPSHOT.write_bytes(response.read())


# Use pandas to read the html returned by the URL - this reads the local file, not the web response
tables = pd.read_html(HELP_SNAPSHOT)

# Expect one table
assert len(tables) == 1, f"expected 1 table on the help page, found {len(tables)}"
endpoints = tables[0]                                   # Only 1 element in the table, a single table

# Ensure the columns of the table are as such
assert list(endpoints.columns) == ["Uri", "Method", "Description"]
endpoints.shape                                         # (70, 3)

endpoints


,Uri,Method,Description
0,/Annual/MonitoringObjective/GroupName={GroupName},GET,This returns the results of annual air quality...
1,/Annual/MonitoringObjective/GroupName={GroupNa...,GET,This returns the results of annual air quality...
2,/Annual/MonitoringObjective/GroupName={GroupNa...,GET,This returns the results of annual air quality...
3,/Annual/MonitoringObjective/GroupName={GroupNa...,GET,This returns the results of annual air quality...
4,/Annual/MonitoringObjective/LocalAuthorityId={...,GET,This returns the results of annual air quality...
...,...,...,...
65,/Information/Species,GET,This lists all of the commonly monitored pollu...
66,/Information/Species/Json,GET,This lists all of the commonly monitored pollu...
67,/Information/Species/SpeciesCode={SpeciesCode},GET,This lists the commonly monitored pollutants f...
68,/Information/Species/SpeciesCode={SpeciesCode}...,GET,This lists the commonly monitored pollutants f...


In [3]:
JsonMask = endpoints["Uri"].str.endswith("/Json")
JsonEndpoints = endpoints[JsonMask].copy()
JsonEndpoints
len(JsonEndpoints)
assert len(JsonEndpoints) == 36

In [14]:
JsonEndpoints["Parameters"] = JsonEndpoints["Uri"].str.findall(r"\{(\w+)\}")
JsonEndpoints[["Uri","Parameters"]]

              


,Uri,Parameters
1,/Annual/MonitoringObjective/GroupName={GroupNa...,[GroupName]
3,/Annual/MonitoringObjective/GroupName={GroupNa...,"[GroupName, Year]"
5,/Annual/MonitoringObjective/LocalAuthorityId={...,"[LocalAuthorityId, Year]"
7,/Annual/MonitoringObjective/SiteCode={SiteCode...,"[SiteCode, StartDate]"
9,/Annual/MonitoringObjective/SiteCode={SiteCode...,"[SiteCode, Year]"
11,/Annual/MonitoringReport/SiteCode={SiteCode}/Y...,"[SiteCode, Year]"
12,/Annualiser/Calculate/year={year}/site={site}/...,"[year, site, backA, backB, backC, backD, speci..."
13,/Annualiser/Sites/year={year}/species={species...,"[year, species]"
15,/Daily/MonitoringIndex/GroupName={GroupName}/D...,"[GroupName, Date]"
17,/Daily/MonitoringIndex/Latest/GroupName={Group...,[GroupName]
